# P2-1 · Bài 1 · Mạng nơ-ron học thế nào

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch
x = torch.randn(128, 3, 32, 32)          # batch ảnh giả
x.shape, x.dtype, x.device               # torch.Size([128, 3, 32, 32]) torch.float32 cpu
device = "cuda" if torch.cuda.is_available() else "cpu"
x = x.to(device)                         # chuyển lên GPU nếu có (model cũng phải ở cùng thiết bị)
x.flatten(1).shape                       # [128, 3072]: duỗi mỗi ảnh thành vector cho MLP

(torch.Size([128, 3, 32, 32]), torch.float32, device(type='cpu'))
torch.Size([128, 3072])


In [ ]:
import torch.nn as nn
model = nn.Sequential(nn.Flatten(), nn.Linear(3072, 256), nn.ReLU(), nn.Linear(256, 10))
sum(p.numel() for p in model.parameters())     # 789258

789258


In [ ]:
x = torch.tensor([1., 2.])
W1 = torch.tensor([[0.5, -0.5], [0.25, 0.75]], requires_grad=True)
w2 = torch.tensor([1., -1.], requires_grad=True)
L = (w2 @ torch.relu(W1 @ x) - 1) ** 2
L.backward()
W1.grad      # tensor([[ 0.0000,  0.0000], [ 5.5000, 11.0000]])
w2.grad      # tensor([-0.0000, -9.6250])

tensor([[ 0.0000,  0.0000],
        [ 5.5000, 11.0000]])
tensor([-0.0000, -9.6250])


In [ ]:
w = torch.tensor(2., requires_grad=True)
for _ in range(2):
    ((3 * w - 1) ** 2).backward()
    print(w.grad)          # 30.0 rồi 60.0, không phải 30.0 hai lần

tensor(30.)
tensor(60.)


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# dữ liệu giả cùng hình dạng CIFAR-10 để chạy thử ngay (CIFAR-10 thật: xem phần thực hành)
# mỗi lớp có một "tông màu" riêng lẫn trong nhiễu mạnh, nên có quy luật để học
palette = torch.randn(10, 3)
def fake_cifar(n):
    y = torch.randint(0, 10, (n,))
    x = torch.randn(n, 3, 32, 32) + 0.08 * palette[y][:, :, None, None]
    return TensorDataset(x, y)
train_loader = DataLoader(fake_cifar(4000), batch_size=128, shuffle=True)
val_loader = DataLoader(fake_cifar(1000), batch_size=256)

model = nn.Sequential(nn.Flatten(), nn.Linear(3072, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()                                  # bật chế độ train (dropout, batchnorm)
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()

    model.eval()                                   # tắt dropout, batchnorm dùng thống kê đã học
    correct = total = 0
    with torch.no_grad():                          # không lưu đồ thị: nhanh hơn, ít bộ nhớ hơn
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item(); total += y.numel()
    print(f"epoch {epoch}: loss {loss.item():.3f}, acc val {correct / total:.3f}")

epoch 0: loss 1.221, acc val 0.577
epoch 1: loss 0.232, acc val 0.599
epoch 2: loss 0.047, acc val 0.600
epoch 3: loss 0.017, acc val 0.596
epoch 4: loss 0.010, acc val 0.595


## Bài tập 4: Bài toán XOR: khi nào cần lớp ẩn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])   # (thành viên, sinh nhật)
y = torch.tensor([[0.], [1.], [1.], [0.]])                    # có tặng quà

def train(model, steps=2000):
    opt = torch.optim.Adam(model.parameters(), lr=0.05)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(steps):
        opt.zero_grad(); loss = loss_fn(model(X), y); loss.backward(); opt.step()
    with torch.no_grad():
        return ((model(X) > 0).float() == y).float().mean().item()
# TODO: acc_logistic, acc_mlp

In [ ]:
torch.manual_seed(0)
acc_logistic = train(nn.Linear(2, 1))
torch.manual_seed(0)
mlp = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
acc_mlp = train(mlp)
print(f"logistic: {acc_logistic:.2f}   mạng có lớp ẩn: {acc_mlp:.2f}")

logistic: 0.75   mạng có lớp ẩn: 1.00


## Bài tập 5: Chuẩn bị batch ảnh cho model {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np, torch
rng = np.random.default_rng(0)
imgs = rng.integers(0, 256, (16, 32, 32, 3), dtype=np.uint8)
imgs[..., 0] = np.clip(imgs[..., 0].astype(int) + 60, 0, 255)       # ảnh hơi ngả đỏ
MEAN = torch.tensor([0.60, 0.50, 0.50])     # tính trước trên tập train
STD = torch.tensor([0.25, 0.29, 0.29])

def to_batch(imgs, mean, std):
    # TODO
    pass

In [ ]:
def to_batch(imgs, mean, std):
    x = torch.from_numpy(imgs).float() / 255          # (N, H, W, C), giá trị 0..1
    x = x.permute(0, 3, 1, 2)                         # (N, C, H, W)
    return (x - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)

xb = to_batch(imgs, MEAN, STD)
print(xb.shape, xb.dtype)
print("trung bình từng kênh:", xb.mean(dim=(0, 2, 3)).round(decimals=2))
print("độ lệch chuẩn từng kênh:", xb.std(dim=(0, 2, 3)).round(decimals=2))

torch.Size([16, 3, 32, 32]) torch.float32
trung bình từng kênh: tensor([0.4200, 0.0100, -0.0000])
độ lệch chuẩn từng kênh: tensor([1.0100, 1.0000, 1.0000])


## Bài tập 6: Nhiệt độ khi chọn từ tiếp theo {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
tu = ["đẹp", "nắng", "lạnh", "mưa", "buồn"]
logits = torch.tensor([3.0, 2.5, 1.0, 0.5, -1.0])
# TODO: probs[T] cho T in (0.5, 1.0, 2.0); so_tu_dung[T]; ce

In [ ]:
import torch.nn.functional as F
torch.manual_seed(0)
probs, so_tu_dung = {}, {}
for T in (0.5, 1.0, 2.0):
    p = torch.softmax(logits / T, dim=0)
    dem = torch.bincount(torch.multinomial(p, 10000, replacement=True), minlength=5)
    probs[T], so_tu_dung[T] = p, int((dem >= 100).sum())
    print(f"T={T}: " + ", ".join(f"{w} {v:.2f}" for w, v in zip(tu, p.tolist())) + f" | {so_tu_dung[T]} từ dùng ≥ 1%")
ce = F.cross_entropy(logits[None], torch.tensor([0])).item()
print(f"cross-entropy (đúng là 'đẹp') = {ce:.3f} = -log({probs[1.0][0]:.3f})")

T=0.5: đẹp 0.72, nắng 0.26, lạnh 0.01, mưa 0.00, buồn 0.00 | 3 từ dùng ≥ 1%
T=1.0: đẹp 0.54, nắng 0.33, lạnh 0.07, mưa 0.04, buồn 0.01 | 4 từ dùng ≥ 1%
T=2.0: đẹp 0.39, nắng 0.30, lạnh 0.14, mưa 0.11, buồn 0.05 | 5 từ dùng ≥ 1%
cross-entropy (đúng là 'đẹp') = 0.611 = -log(0.543)


## Bài tập 7: Chọn learning rate cho bài toán chữ số {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
from sklearn.datasets import load_digits
d = load_digits()
X = torch.tensor(d.data / 16, dtype=torch.float32)
y = torch.tensor(d.target)

def thu(lr, steps=300):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad(); loss = nn.functional.cross_entropy(model(X), y); loss.backward(); opt.step()
    return loss.item()
# TODO: ket_qua = {lr: loss cuối}; lr_tot

In [ ]:
import math
ket_qua = {lr: thu(lr) for lr in [0.0001, 0.01, 0.1, 1, 10]}
for lr, l in ket_qua.items():
    print(f"lr = {lr:<7}: loss cuối {l:.4f}")
lr_tot = min((lr for lr in ket_qua if math.isfinite(ket_qua[lr])), key=ket_qua.get)
print("tốt nhất:", lr_tot)

lr = 0.0001 : loss cuối 2.3247
lr = 0.01   : loss cuối 2.1729
lr = 0.1    : loss cuối 0.3183
lr = 1      : loss cuối 0.0425
lr = 10     : loss cuối 2.3025
tốt nhất: 1


## Bài tập 8: Vòng train đầy đủ cho chữ số viết tay {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.datasets import load_digits
d = load_digits()
ds = TensorDataset(torch.tensor(d.data / 16, dtype=torch.float32), torch.tensor(d.target))
torch.manual_seed(0)
train_ds, val_ds = random_split(ds, [1437, 360])
# TODO: loader, model, opt, vòng 15 epoch, acc (accuracy validation cuối)

In [ ]:
loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_X = torch.stack([x for x, _ in val_ds]); val_y = torch.stack([t for _, t in val_ds])
model = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(1, 16):
    model.train(); tong = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = nn.functional.cross_entropy(model(xb), yb)
        loss.backward(); opt.step()
        tong += loss.item() * len(xb)
    model.eval()
    with torch.no_grad():
        acc = (model(val_X).argmax(1) == val_y).float().mean().item()
    if epoch % 3 == 0:
        print(f"epoch {epoch:2d}: loss train {tong / len(train_ds):.3f}, val acc {acc:.3f}")

epoch  3: loss train 1.258, val acc 0.878
epoch  6: loss train 0.474, val acc 0.925
epoch  9: loss train 0.286, val acc 0.942
epoch 12: loss train 0.209, val acc 0.947
epoch 15: loss train 0.167, val acc 0.956
